
# Detecção de Intrusões em Redes com Deep Learning

**Tema:** aplicação de Deep Learning à segurança cibernética  
**Problema:** classificar tráfego de rede como **normal (0)** ou **ataque (1)**.  
**Base:** **UNSW-NB15**, obtida do Kaggle.

O notebook implementa uma **Deep Neural Network (DNN/MLP)** com TensorFlow/Keras e avalia o modelo com métricas adequadas para detecção de intrusões: **precision, recall, F1-score, ROC-AUC e matriz de confusão**.

> **Pergunta de pesquisa:** uma rede neural profunda consegue aprender padrões do tráfego de rede e identificar automaticamente conexões potencialmente maliciosas?

### Fluxo do projeto

1. Baixar a base do Kaggle.
2. Explorar os dados e a distribuição das classes.
3. Separar variáveis de entrada e alvo.
4. Tratar dados numéricos e categóricos.
5. Separar treino, validação e teste.
6. Treinar uma rede neural profunda.
7. Ajustar o limiar de decisão com a validação.
8. Avaliar o modelo no conjunto de teste.
9. Interpretar resultados e limitações.



## 1. Instalação das bibliotecas

O `kagglehub` é usado para baixar a base pública diretamente do Kaggle.  
Se o Colab solicitar autenticação, basta seguir a mensagem exibida pelo próprio Kaggle.


In [ ]:

!pip -q install kagglehub pyarrow


In [ ]:

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)



## 2. Download da base no Kaggle

Dataset utilizado: **UNSW-NB15** (`mrwellsdavid/unsw-nb15`).

A base foi criada para pesquisa em sistemas de detecção de intrusões e contém registros de tráfego normal e diferentes tipos de ataque.


In [ ]:

import kagglehub

dataset_path = kagglehub.dataset_download("mrwellsdavid/unsw-nb15")
dataset_path = Path(dataset_path)

print("Base baixada em:", dataset_path)

all_files = sorted([p for p in dataset_path.rglob("*") if p.is_file()])
print("\nArquivos encontrados:")
for p in all_files:
    print("-", p.name)



## 3. Carregamento dos conjuntos de treino e teste

O código abaixo procura automaticamente pelos arquivos de **training-set** e **testing-set**, aceitando tanto CSV quanto Parquet.


In [ ]:

def choose_split_file(files, split):
    split = split.lower()
    valid = [p for p in files if p.suffix.lower() in {".csv", ".parquet"}]

    if split == "train":
        keywords = ["training-set", "training_set", "train"]
    else:
        keywords = ["testing-set", "testing_set", "test"]

    candidates = []
    for p in valid:
        name = p.name.lower()
        if any(k in name for k in keywords):
            candidates.append(p)

    if not candidates:
        raise FileNotFoundError(
            f"Não foi possível localizar automaticamente o arquivo de {split}. "
            f"Arquivos disponíveis: {[p.name for p in valid]}"
        )

    # Prefere parquet quando houver, por ser mais rápido e compacto.
    candidates = sorted(
        candidates,
        key=lambda p: (0 if p.suffix.lower() == ".parquet" else 1, len(p.name))
    )
    return candidates[0]

def read_table(path):
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)

train_file = choose_split_file(all_files, "train")
test_file = choose_split_file(all_files, "test")

print("Treino:", train_file.name)
print("Teste :", test_file.name)

train_df = read_table(train_file)
test_df = read_table(test_file)

print("\nDimensão treino:", train_df.shape)
print("Dimensão teste :", test_df.shape)

display(train_df.head())



## 4. Entendimento inicial da base

O alvo principal é `label`:

- `0` = tráfego normal
- `1` = ataque

A coluna `attack_cat`, quando presente, informa o tipo de ataque. Como ela praticamente revela a natureza do evento, **não será utilizada como variável de entrada** no problema binário, evitando vazamento de informação (*data leakage*).


In [ ]:

print("Colunas:")
print(train_df.columns.tolist())

print("\nTipos de dados:")
display(train_df.dtypes.to_frame("dtype"))

if "label" not in train_df.columns:
    raise ValueError("A coluna alvo 'label' não foi encontrada na base.")

print("\nDistribuição da variável alvo - treino:")
display(
    train_df["label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .to_frame("quantidade")
    .assign(percentual=lambda x: 100 * x["quantidade"] / x["quantidade"].sum())
)

if "attack_cat" in train_df.columns:
    print("\nPrincipais categorias de ataque - treino:")
    display(train_df["attack_cat"].value_counts(dropna=False).head(15).to_frame("quantidade"))


In [ ]:

# Visualização simples da distribuição das classes
class_counts = train_df["label"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["Normal (0)", "Ataque (1)"], class_counts.values)
plt.title("Distribuição das classes no conjunto de treino")
plt.ylabel("Quantidade de registros")
plt.show()



## 5. Preparação dos dados

Serão removidas:

- `label`: variável alvo;
- `attack_cat`: categoria do ataque, para evitar vazamento de informação;
- `id`: identificador do registro, quando existente.

Depois:

- variáveis numéricas → imputação pela mediana + padronização;
- variáveis categóricas → imputação pela moda + One-Hot Encoding.

O conjunto oficial de treino é ainda dividido em **treino interno (80%)** e **validação (20%)**.  
O conjunto oficial de teste permanece separado até a avaliação final.


In [ ]:

train_df = train_df.replace([np.inf, -np.inf], np.nan).copy()
test_df = test_df.replace([np.inf, -np.inf], np.nan).copy()

# Garante que a variável alvo seja numérica/binária.
train_df = train_df.dropna(subset=["label"]).copy()
test_df = test_df.dropna(subset=["label"]).copy()

train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

drop_features = [c for c in ["label", "attack_cat", "id"] if c in train_df.columns]

X_full = train_df.drop(columns=drop_features)
y_full = train_df["label"]

X_test_raw = test_df.drop(columns=[c for c in ["label", "attack_cat", "id"] if c in test_df.columns])
y_test = test_df["label"]

# Alinha as colunas do teste com as do treino, caso necessário.
X_test_raw = X_test_raw.reindex(columns=X_full.columns)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    stratify=y_full,
    random_state=SEED,
)

numeric_features = X_train_raw.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_features = [c for c in X_train_raw.columns if c not in numeric_features]

print("Variáveis numéricas:", len(numeric_features))
print("Variáveis categóricas:", len(categorical_features))
print("\nTreino interno:", X_train_raw.shape)
print("Validação     :", X_val_raw.shape)
print("Teste final   :", X_test_raw.shape)


In [ ]:

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                dtype=np.float32,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

X_train = preprocessor.fit_transform(X_train_raw).astype(np.float32)
X_val = preprocessor.transform(X_val_raw).astype(np.float32)
X_test = preprocessor.transform(X_test_raw).astype(np.float32)

print("Dimensão após pré-processamento:")
print("Treino    :", X_train.shape)
print("Validação :", X_val.shape)
print("Teste     :", X_test.shape)



## 6. Peso das classes

Em problemas de segurança cibernética, errar um ataque pode ter custo diferente de errar um tráfego normal.  
Por isso, o treinamento utiliza pesos calculados automaticamente a partir da frequência de cada classe.


In [ ]:

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print("Pesos das classes:", class_weight)



## 7. Construção da Rede Neural Profunda

Arquitetura escolhida:

- entrada com todas as variáveis após o pré-processamento;
- camada densa com 128 neurônios;
- camada densa com 64 neurônios;
- camada densa com 32 neurônios;
- regularização com **Dropout**;
- saída com 1 neurônio e função **sigmoid**.

A função de perda é `binary_crossentropy`, adequada para classificação binária.


In [ ]:

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.30),

    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.30),

    layers.Dense(32, activation="relu"),
    layers.Dropout(0.20),

    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ],
)

model.summary()



## 8. Treinamento

O `EarlyStopping` interrompe o treinamento quando a perda de validação deixa de melhorar, reduzindo o risco de *overfitting*.


In [ ]:

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1,
    ),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=256,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)



## 9. Curvas de treinamento


In [ ]:

history_df = pd.DataFrame(history.history)

plt.figure(figsize=(7, 4))
plt.plot(history_df["loss"], label="Treino")
plt.plot(history_df["val_loss"], label="Validação")
plt.title("Loss ao longo das épocas")
plt.xlabel("Época")
plt.ylabel("Binary Cross-Entropy")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history_df["accuracy"], label="Treino")
plt.plot(history_df["val_accuracy"], label="Validação")
plt.title("Acurácia ao longo das épocas")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.legend()
plt.show()



## 10. Escolha do limiar de decisão

A rede retorna uma probabilidade entre 0 e 1.  
Em vez de assumir automaticamente o limiar `0.50`, buscamos no conjunto de validação o limiar com melhor **F1-score**.

Isso é especialmente útil em detecção de ataques, pois permite equilibrar **precision** e **recall** sem usar o conjunto de teste para tomar decisões.


In [ ]:

val_prob = model.predict(X_val, verbose=0).ravel()

thresholds = np.arange(0.10, 0.91, 0.01)
f1_values = []

for t in thresholds:
    pred_t = (val_prob >= t).astype(int)
    f1_values.append(f1_score(y_val, pred_t))

best_idx = int(np.argmax(f1_values))
best_threshold = float(thresholds[best_idx])
best_val_f1 = float(f1_values[best_idx])

print(f"Melhor limiar na validação: {best_threshold:.2f}")
print(f"F1 de validação nesse limiar: {best_val_f1:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(thresholds, f1_values)
plt.axvline(best_threshold, linestyle="--", label=f"Melhor limiar = {best_threshold:.2f}")
plt.title("F1-score por limiar de decisão")
plt.xlabel("Limiar")
plt.ylabel("F1-score")
plt.legend()
plt.show()



## 11. Avaliação final no conjunto de teste

O conjunto de teste não foi usado para treinamento, pré-processamento ou escolha do limiar.


In [ ]:

test_prob = model.predict(X_test, verbose=0).ravel()
test_pred = (test_prob >= best_threshold).astype(int)

metrics = {
    "Acurácia": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0),
    "F1-score": f1_score(y_test, test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, test_prob),
}

metrics_df = pd.DataFrame.from_dict(metrics, orient="index", columns=["Valor"])
display(metrics_df.style.format("{:.4f}"))

print("\nRelatório de classificação:")
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["Normal", "Ataque"],
        digits=4,
        zero_division=0,
    )
)



## 12. Matriz de confusão

Na segurança cibernética, um ponto especialmente importante é o **falso negativo**:

> ataque real que o modelo classifica como tráfego normal.

Esse tipo de erro pode permitir que uma atividade maliciosa passe pelo sistema de detecção.


In [ ]:

cm = confusion_matrix(y_test, test_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Ataque"],
)
disp.plot(values_format="d")
plt.title("Matriz de confusão - conjunto de teste")
plt.show()

tn, fp, fn, tp = cm.ravel()

print(f"Verdadeiros negativos (normal corretamente identificado): {tn}")
print(f"Falsos positivos (normal marcado como ataque):           {fp}")
print(f"Falsos negativos (ataque não detectado):                 {fn}")
print(f"Verdadeiros positivos (ataque corretamente detectado):   {tp}")



## 13. Curvas ROC e Precision-Recall


In [ ]:

RocCurveDisplay.from_predictions(y_test, test_prob)
plt.title("Curva ROC")
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, test_prob)
plt.title("Curva Precision-Recall")
plt.show()



## 14. Leitura dos resultados

Para a apresentação, evite interpretar apenas a acurácia.

### Métricas principais

- **Precision:** entre os registros classificados como ataque, quantos realmente eram ataques.
- **Recall:** entre todos os ataques existentes, quantos foram detectados.
- **F1-score:** média harmônica entre precision e recall.
- **ROC-AUC:** capacidade do modelo de separar as duas classes considerando diferentes limiares.

### Por que o Recall é importante?

Em um sistema de detecção de intrusões, um **falso negativo** significa que um ataque ocorreu, mas não foi detectado.  
Por isso, recall e F1-score são especialmente relevantes para analisar o modelo.

### O que o modelo faz de fato?

A DNN aprende relações não lineares entre características do tráfego de rede, como protocolos, estado da conexão, volume de bytes, pacotes, taxas e outras informações, e produz uma probabilidade de o registro representar uma atividade maliciosa.



## 15. Limitações

Mesmo que o modelo alcance boas métricas, ele **não deve ser interpretado como um sistema de segurança pronto para produção**.

Principais limitações:

1. A base representa um conjunto específico de cenários e ataques.
2. Ameaças cibernéticas mudam ao longo do tempo (*concept drift*).
3. Um ambiente real pode apresentar padrões de tráfego diferentes.
4. Falsos positivos excessivos podem gerar sobrecarga para equipes de segurança.
5. O modelo deve ser validado continuamente com dados recentes.
6. Deep Learning não elimina a necessidade de políticas, controles e especialistas em segurança.

---

## 16. Conclusão sugerida para o trabalho

> A implementação mostrou como técnicas de Deep Learning podem ser aplicadas à detecção de intrusões em redes. A rede neural foi treinada para reconhecer padrões presentes nas características do tráfego e classificar cada registro como normal ou potencialmente malicioso. A análise por precision, recall, F1-score, ROC-AUC e matriz de confusão permite avaliar não apenas a taxa geral de acertos, mas também os erros mais críticos do ponto de vista de segurança. Apesar do potencial da abordagem, sua aplicação real exige atualização contínua dos dados, controle de falsos positivos e validação em ambientes de produção.



## 17. Roteiro enxuto para apresentação

**Slide 1 — Problema**  
Ataques em redes precisam ser identificados rapidamente; análise manual não escala.

**Slide 2 — Objetivo**  
Implementar uma rede neural profunda capaz de classificar tráfego como normal ou ataque.

**Slide 3 — Base de dados**  
UNSW-NB15, obtida do Kaggle, com características de tráfego de rede e rótulos de segurança.

**Slide 4 — Tratamento dos dados**  
Limpeza, separação treino/validação/teste, padronização e One-Hot Encoding.

**Slide 5 — Arquitetura de Deep Learning**  
DNN com camadas 128 → 64 → 32 → 1, ReLU, Dropout e Sigmoid.

**Slide 6 — Resultados**  
Mostrar F1-score, Recall, ROC-AUC e matriz de confusão obtidos após executar o notebook.

**Slide 7 — Conclusão e limitações**  
Deep Learning consegue aprender padrões de intrusão, mas implantação real exige monitoramento, atualização e análise de falsos positivos/falsos negativos.
